# Member 1 – Data engineering & feature preparation

**Goal:** turn `final_dataset.csv` (10 raw variables) into four clean, ready-to-use tables:

| File | Content |
|---|---|
| `features_ml1.parquet` / `targets_ml1.parquet` | SPY / market-regime model |
| `features_ml2.parquet` / `targets_ml2.parquet` | 4-ETF volatility model |

**Conventions**
- Prices are assumed to be *adjusted* close prices (as downloaded). Returns are simple daily returns.
- Volatility = rolling std of daily returns × √252 (annualised). The 1-day "volatility" is |return| × √252 (a std over one observation is undefined).
- Feature at date *t* only uses data ≤ *t*. Target at date *t* only uses data > *t* (`shift(-h)`).
- Rows with any NaN (warm-up at the start, no future at the end) are dropped, so the ML members get tables with no NaN.

In [5]:
!pip install pandas numpy pyarrow   # uncomment if needed
import shutil
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

# ---------- Configuration ----------
SRC_CSV   = Path("../src/data/financial_dataset/processed/final_dataset.csv")
RAW_DIR   = Path("../src/data/financial_dataset/raw")
PROC_DIR  = Path("../src/data/financial_dataset/processed")
ETFS      = ["SPY", "IEF", "LQD", "VNQ"]
MACRO     = ["VIX", "BAA10Y", "DGS3MO", "DGS2", "DGS10", "T10Y2Y"]
ANN       = np.sqrt(252)                   # annualisation factor
HORIZON   = 21                             # forecast horizon (trading days) for the targets
STRESS_VOL_THRESHOLD = 0.20                # annualised fwd vol above which regime = "stress" (to agree with ML1 member)

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


## 1. Load `final_dataset.csv`

In [6]:
df = pd.read_csv(SRC_CSV)
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

missing_cols = set(ETFS + MACRO) - set(df.columns)
assert not missing_cols, f"Missing columns: {missing_cols}"

# keep a copy of the original file in data/raw
shutil.copy(SRC_CSV, RAW_DIR / SRC_CSV.name)

print(df.shape, df["Date"].min().date(), "->", df["Date"].max().date())
df.head()

(5517, 11) 2004-09-29 -> 2026-11-04


,Date,SPY,IEF,LQD,VNQ,VIX,BAA10Y,DGS3MO,DGS2,DGS10,T10Y2Y
0,2004-09-29,74.703812,47.323967,45.991653,19.328249,13.21,2.12,1.72,2.61,4.10,1.49
1,2004-09-30,74.650368,47.274479,45.856255,19.483341,13.34,2.11,1.71,2.63,4.14,1.51
2,2004-10-01,75.912796,47.085064,45.653194,19.843931,12.75,2.09,1.71,2.63,4.21,1.58
3,2004-10-04,76.039726,47.101643,45.624393,19.929237,13.41,2.10,1.71,2.65,4.19,1.54
4,2004-10-05,76.079781,47.101643,45.813847,19.925362,13.95,2.10,1.71,2.65,4.18,1.53


## 2. Basic cleaning
- Duplicate dates → keep last.
- Days where an ETF price is missing = market closed → dropped.
- FRED series (VIX aside, BAA10Y, DGS*, T10Y2Y) have gaps on bank holidays → forward-filled (max 5 days, past values only, so no leakage).

In [7]:
n_dup = df.duplicated("Date").sum()
print("Duplicate dates:", n_dup)
df = df.drop_duplicates("Date", keep="last").set_index("Date").sort_index()

print("\nMissing values BEFORE cleaning:")
print(df.isna().sum())

df = df.dropna(subset=ETFS)                         # non-trading days
df[MACRO] = df[MACRO].ffill(limit=5)                # past values only

print("\nMissing values AFTER cleaning:")
print(df.isna().sum())
assert df.index.is_monotonic_increasing and df.index.is_unique
print("\nRange check:")
df.describe().T[["min", "max", "mean"]]

Duplicate dates: 0

Missing values BEFORE cleaning:
SPY       0
IEF       0
LQD       0
VNQ       0
VIX       0
BAA10Y    0
DGS3MO    0
DGS2      0
DGS10     0
T10Y2Y    0
dtype: int64

Missing values AFTER cleaning:
SPY       0
IEF       0
LQD       0
VNQ       0
VIX       0
BAA10Y    0
DGS3MO    0
DGS2      0
DGS10     0
T10Y2Y    0
dtype: int64

Range check:


,min,max,mean
SPY,49.557499,779.090027,241.169999
IEF,46.386059,104.941322,77.388641
LQD,41.109661,111.469414,78.052501
VNQ,10.363995,100.081970,51.859165
VIX,9.140000,82.690000,19.032686
BAA10Y,1.360000,6.160000,2.390210
DGS3MO,0.000000,5.630000,1.843950
DGS2,0.090000,5.290000,2.088242
DGS10,0.520000,5.310000,3.046529
T10Y2Y,-1.080000,2.910000,0.958287


**Credit spread validation (BAA10Y).** Quick sanity checks before using it as a feature: share of missing values and plausible range (Moody's Baa – 10Y Treasury, in % points, typically 1.5–6).

In [8]:
baa = df["BAA10Y"]
baa_valid = (baa.isna().mean() < 0.05) and baa.dropna().between(0, 10).all()
print(f"BAA10Y missing: {baa.isna().mean():.2%} | min {baa.min():.2f} | max {baa.max():.2f} | valid: {baa_valid}")
USE_BAA10Y = bool(baa_valid)

BAA10Y missing: 0.00% | min 1.36 | max 6.16 | valid: True


## 3. Feature engineering
One function builds **all** features from the cleaned data. Wrapping it in a function lets us re-run it on truncated data later to prove there is no look-ahead (section 6).

In [9]:
def build_features(d: pd.DataFrame) -> pd.DataFrame:
    px  = d[ETFS]
    ret = px.pct_change()                                   # daily returns (adjusted prices)
    f = pd.DataFrame(index=d.index)

    # --- per-ETF returns and volatilities ---
    for t in ETFS:
        f[f"{t}_ret_1d"] = ret[t]
        for w in (5, 21, 63):
            f[f"{t}_ret_{w}d"] = px[t].pct_change(w)
        f[f"{t}_vol_1d"] = ret[t].abs() * ANN              # 1-day proxy
        for w in (5, 21, 63):
            f[f"{t}_vol_{w}d"] = ret[t].rolling(w).std() * ANN

    # --- VIX ---
    f["VIX"]        = d["VIX"]
    f["VIX_chg_1d"] = d["VIX"].pct_change(1)
    f["VIX_chg_5d"] = d["VIX"].pct_change(5)

    # --- SPY trend / drawdown ---
    f["SPY_drawdown_252d"] = px["SPY"] / px["SPY"].rolling(252).max() - 1
    f["SPY_vs_ma200"]      = px["SPY"] / px["SPY"].rolling(200).mean() - 1

    # --- macro ---
    f["T10Y2Y"] = d["T10Y2Y"]
    if USE_BAA10Y:
        f["BAA10Y"] = d["BAA10Y"]

    # --- cross-asset features ---
    f["SPY_IEF_corr_63d"] = ret["SPY"].rolling(63).corr(ret["IEF"])
    f["SPY_LQD_corr_63d"] = ret["SPY"].rolling(63).corr(ret["LQD"])
    f["SPY_VNQ_corr_63d"] = ret["SPY"].rolling(63).corr(ret["VNQ"])
    f["IEF_LQD_corr_63d"] = ret["IEF"].rolling(63).corr(ret["LQD"])
    f["vol_ratio_SPY_IEF_21d"] = f["SPY_vol_21d"] / f["IEF_vol_21d"]
    f["vol_ratio_VNQ_SPY_21d"] = f["VNQ_vol_21d"] / f["SPY_vol_21d"]
    f["vol_ratio_LQD_IEF_21d"] = f["LQD_vol_21d"] / f["IEF_vol_21d"]
    f["mean_vol_21d"]          = f[[f"{t}_vol_21d" for t in ETFS]].mean(axis=1)
    f["vol_5d_over_63d_SPY"]   = f["SPY_vol_5d"] / f["SPY_vol_63d"]

    return f.replace([np.inf, -np.inf], np.nan)


def select_ml1(f: pd.DataFrame) -> pd.DataFrame:
    cols  = [f"SPY_ret_{w}d" for w in (1, 5, 21, 63)]
    cols += [f"SPY_vol_{w}d" for w in (5, 21, 63)]
    cols += ["VIX", "VIX_chg_5d", "SPY_drawdown_252d", "SPY_vs_ma200", "T10Y2Y", "SPY_IEF_corr_63d"]
    if USE_BAA10Y:
        cols.append("BAA10Y")
    return f[cols]


def select_ml2(f: pd.DataFrame) -> pd.DataFrame:
    cols  = [f"{t}_vol_{w}d" for t in ETFS for w in (1, 5, 21, 63)]
    cols += [f"{t}_ret_{w}d" for t in ETFS for w in (1, 5, 21, 63)]
    cols += ["VIX", "VIX_chg_1d", "VIX_chg_5d"]
    cols += ["SPY_IEF_corr_63d", "SPY_LQD_corr_63d", "SPY_VNQ_corr_63d", "IEF_LQD_corr_63d",
             "vol_ratio_SPY_IEF_21d", "vol_ratio_VNQ_SPY_21d", "vol_ratio_LQD_IEF_21d", "mean_vol_21d"]
    return f[cols]


feat_all = build_features(df)
feat1_raw, feat2_raw = select_ml1(feat_all), select_ml2(feat_all)
print("ML1 features:", feat1_raw.shape, "| ML2 features:", feat2_raw.shape)
feat1_raw.tail(3)

ML1 features: (5517, 14) | ML2 features: (5517, 43)


,SPY_ret_1d,SPY_ret_5d,SPY_ret_21d,SPY_ret_63d,SPY_vol_5d,SPY_vol_21d,SPY_vol_63d,VIX,VIX_chg_5d,SPY_drawdown_252d,SPY_vs_ma200,T10Y2Y,SPY_IEF_corr_63d,BAA10Y
Date,,,,,,,,,,,,,,
2026-11-02,-0.004790,-0.017338,-0.015563,-0.004887,0.145348,0.117521,0.104982,21.15,0.150082,-0.034643,0.036158,0.32,0.181483,1.53
2026-11-03,0.002992,-0.019892,-0.019866,-0.019585,0.135962,0.114644,0.098565,20.83,0.188927,-0.031755,0.038775,0.34,0.123405,1.54
2026-11-04,0.016518,-0.007121,-0.010351,-0.001398,0.198321,0.127200,0.104068,16.61,-0.017741,-0.015762,0.055348,0.32,0.179379,1.52


## 4. Targets (what the models must predict)
All targets look **forward** from date *t* over the next `HORIZON` = 21 trading days (t+1 … t+21).

**ML1 (SPY / regime)**
- `y_SPY_vol_21d_fwd` – realised volatility over the next 21 days (annualised)
- `y_SPY_ret_21d_fwd` – cumulative return over the next 21 days
- `y_SPY_maxdd_21d_fwd` – worst peak-to-date loss vs. today's price over the next 21 days
- `y_stress_regime` – 1 if `y_SPY_vol_21d_fwd` > threshold (fixed, not data-derived, so no leakage), else 0

**ML2 (volatility of the 4 ETFs)**
- `y_<ETF>_vol_21d_fwd` and `y_<ETF>_ret_21d_fwd` for SPY, IEF, LQD, VNQ

In [10]:
def fwd_vol(ret: pd.Series, h: int = HORIZON) -> pd.Series:
    # rolling std at t+h covers returns t+1..t+h
    return ret.rolling(h).std().shift(-h) * ANN


def fwd_ret(px: pd.Series, h: int = HORIZON) -> pd.Series:
    return px.shift(-h) / px - 1


def fwd_maxdd(px: pd.Series, h: int = HORIZON) -> pd.Series:
    # minimum of the prices t+1..t+h, relative to price at t
    fwd_min = px.shift(-1)[::-1].rolling(h).min()[::-1]
    return fwd_min / px - 1


def build_targets(d: pd.DataFrame):
    px  = d[ETFS]
    ret = px.pct_change()

    t1 = pd.DataFrame(index=d.index)
    t1["y_SPY_vol_21d_fwd"]   = fwd_vol(ret["SPY"])
    t1["y_SPY_ret_21d_fwd"]   = fwd_ret(px["SPY"])
    t1["y_SPY_maxdd_21d_fwd"] = fwd_maxdd(px["SPY"])
    t1["y_stress_regime"]     = (t1["y_SPY_vol_21d_fwd"] > STRESS_VOL_THRESHOLD).astype(float)
    t1.loc[t1["y_SPY_vol_21d_fwd"].isna(), "y_stress_regime"] = np.nan

    t2 = pd.DataFrame(index=d.index)
    for t in ETFS:
        t2[f"y_{t}_vol_21d_fwd"] = fwd_vol(ret[t])
        t2[f"y_{t}_ret_21d_fwd"] = fwd_ret(px[t])
    return t1, t2


targ1_raw, targ2_raw = build_targets(df)
print("Stress-regime share:", targ1_raw["y_stress_regime"].mean().round(3))
targ1_raw.dropna().head(3)

Stress-regime share: 0.197


,y_SPY_vol_21d_fwd,y_SPY_ret_21d_fwd,y_SPY_maxdd_21d_fwd,y_stress_regime
Date,,,,
2004-09-29,0.129589,0.012160,-0.017704,0.0
2004-09-30,0.129697,0.015659,-0.017001,0.0
2004-10-01,0.115659,-0.000880,-0.033348,0.0


## 5. Align features & targets and drop NaN rows (warm-up at the start, no future at the end)

In [11]:
def align(feat, targ):
    idx = feat.dropna().index.intersection(targ.dropna().index)
    return feat.loc[idx].copy(), targ.loc[idx].copy()

features_ml1, targets_ml1 = align(feat1_raw, targ1_raw)
features_ml2, targets_ml2 = align(feat2_raw, targ2_raw)
targets_ml1["y_stress_regime"] = targets_ml1["y_stress_regime"].astype(int)

for name, f, t in [("ML1", features_ml1, targets_ml1), ("ML2", features_ml2, targets_ml2)]:
    print(f"{name}: {f.shape[0]} rows | {f.index.min().date()} -> {f.index.max().date()} | "
          f"{f.shape[1]} features | {t.shape[1]} targets")

ML1: 5245 rows | 2005-09-29 -> 2026-10-05 | 14 features | 4 targets
ML2: 5433 rows | 2004-12-31 -> 2026-10-05 | 43 features | 8 targets


## 6. Quality checks

### 6.1 Missing values & duplicate dates

In [12]:
for name, tbl in [("features_ml1", features_ml1), ("targets_ml1", targets_ml1),
                  ("features_ml2", features_ml2), ("targets_ml2", targets_ml2)]:
    assert tbl.isna().sum().sum() == 0, f"NaN in {name}"
    assert tbl.index.is_unique and tbl.index.is_monotonic_increasing, f"Index problem in {name}"
    assert not np.isinf(tbl.select_dtypes("number")).any().any(), f"inf in {name}"
assert features_ml1.index.equals(targets_ml1.index) and features_ml2.index.equals(targets_ml2.index)
print("OK: no NaN, no inf, no duplicate dates, features/targets perfectly aligned.")

OK: no NaN, no inf, no duplicate dates, features/targets perfectly aligned.


### 6.2 Extreme returns
Extreme moves are **flagged, not removed** (e.g. March 2020 is real information the models must see). Check that they are real events and not data errors.

In [13]:
rets = df[ETFS].pct_change()
z = (rets - rets.mean()) / rets.std()
extreme = rets[(rets.abs() > 0.08) | (z.abs() > 8)].dropna(how="all")
print(f"{len(extreme)} days with |ret| > 8% or |z| > 8")
extreme.round(4)

60 days with |ret| > 8% or |z| > 8


,SPY,IEF,LQD,VNQ
Date,,,,
2008-01-23,NaN,NaN,NaN,0.0949
2008-09-16,NaN,NaN,-0.0635,NaN
2008-09-17,NaN,NaN,-0.0458,NaN
2008-09-18,NaN,NaN,NaN,0.0850
2008-09-22,NaN,NaN,NaN,-0.0840
2008-09-29,NaN,NaN,-0.0911,NaN
2008-09-30,NaN,NaN,0.0977,NaN
2008-10-07,NaN,NaN,NaN,-0.0858
2008-10-10,NaN,NaN,-0.0551,0.0957


### 6.3 Future-data leakage
Two tests:
1. **Truncation test (features):** recompute the features using only data up to a random date *t*. The last row must equal the row of the full-data computation. If any feature peeked at the future, it would differ.
2. **Manual target check:** recompute a target by hand for random dates from the raw prices, using only t+1…t+21.

In [14]:
rng = np.random.default_rng(42)
cut_dates = rng.choice(features_ml2.index[:-HORIZON], size=15, replace=False)

for cut in cut_dates:
    f_trunc = build_features(df.loc[:cut])
    row_full  = feat_all.loc[cut]
    row_trunc = f_trunc.loc[cut]
    pd.testing.assert_series_equal(row_full, row_trunc, check_names=False, rtol=1e-9, atol=1e-12)
print("OK: features at t are identical when the future is removed -> no look-ahead in features.")

# manual target check
ret_spy = df["SPY"].pct_change()
for t in rng.choice(features_ml1.index[:-HORIZON], size=5, replace=False):
    pos = df.index.get_loc(t)
    window = ret_spy.iloc[pos + 1 : pos + 1 + HORIZON]
    manual_vol = window.std() * ANN
    manual_ret = df["SPY"].iloc[pos + HORIZON] / df["SPY"].iloc[pos] - 1
    manual_dd  = df["SPY"].iloc[pos + 1 : pos + 1 + HORIZON].min() / df["SPY"].iloc[pos] - 1
    assert np.isclose(targets_ml1.loc[t, "y_SPY_vol_21d_fwd"], manual_vol)
    assert np.isclose(targets_ml1.loc[t, "y_SPY_ret_21d_fwd"], manual_ret)
    assert np.isclose(targets_ml1.loc[t, "y_SPY_maxdd_21d_fwd"], manual_dd)
print("OK: targets use only t+1 ... t+21.")

OK: features at t are identical when the future is removed -> no look-ahead in features.
OK: targets use only t+1 ... t+21.


### 6.4 Overview
Note for ML members: consecutive rows have **overlapping 21-day target windows** (strong autocorrelation). Use time-ordered splits (train < validation < test), never random shuffling, and leave a gap of ≥ 21 days between train and validation/test.

In [15]:
display(features_ml1.describe().T.round(4))
display(targets_ml1.describe().T.round(4))

,count,mean,std,min,25%,50%,75%,max
SPY_ret_1d,5245.0,0.0005,0.0121,-0.1094,-0.0039,0.0007,0.0059,0.1282
SPY_ret_5d,5245.0,0.0024,0.0243,-0.1979,-0.0081,0.0041,0.0150,0.1940
SPY_ret_21d,5245.0,0.0100,0.0464,-0.3275,-0.0104,0.0160,0.0361,0.2519
SPY_ret_63d,5245.0,0.0299,0.0759,-0.4052,-0.0043,0.0405,0.0745,0.3994
SPY_vol_5d,5245.0,0.1490,0.1269,0.0068,0.0747,0.1180,0.1827,1.4471
SPY_vol_21d,5245.0,0.1580,0.1105,0.0330,0.0941,0.1313,0.1840,0.9445
SPY_vol_63d,5245.0,0.1638,0.1000,0.0511,0.1091,0.1333,0.1858,0.7394
VIX,5245.0,19.3260,8.5707,9.1400,13.7400,17.0100,22.2400,82.6900
VIX_chg_5d,5245.0,0.0122,0.1678,-0.4667,-0.0815,-0.0092,0.0778,2.1290
SPY_drawdown_252d,5245.0,-0.0522,0.0837,-0.5148,-0.0635,-0.0178,-0.0031,0.0000


,count,mean,std,min,25%,50%,75%,max
y_SPY_vol_21d_fwd,5245.0,0.1579,0.1105,0.0330,0.0941,0.1311,0.1840,0.9445
y_SPY_ret_21d_fwd,5245.0,0.0101,0.0463,-0.3275,-0.0101,0.0160,0.0361,0.2519
y_SPY_maxdd_21d_fwd,5245.0,-0.0263,0.0388,-0.3275,-0.0373,-0.0153,-0.0029,0.0906
y_stress_regime,5245.0,0.2059,0.4044,0.0000,0.0000,0.0000,0.0000,1.0000


## 7. Save to Parquet

In [18]:
features_ml1.to_parquet(PROC_DIR / "features_ml1.parquet")
targets_ml1.to_parquet(PROC_DIR / "targets_ml1.parquet")
features_ml2.to_parquet(PROC_DIR / "features_ml2.parquet")
targets_ml2.to_parquet(PROC_DIR / "targets_ml2.parquet")

# Definition of done: reload with plain pandas, nothing else to clean
for name in ["features_ml1", "targets_ml1", "features_ml2", "targets_ml2"]:
    t = pd.read_parquet(PROC_DIR / f"{name}.parquet")
    print(f"{name:14s} shape={t.shape}  NaN={int(t.isna().sum().sum())}  index={t.index.name}")

print("\nFiles:")
for p in sorted(Path("data").rglob("*")):
    if p.is_file():
        print(" ", p)

features_ml1   shape=(5245, 14)  NaN=0  index=Date
targets_ml1    shape=(5245, 4)  NaN=0  index=Date
features_ml2   shape=(5433, 43)  NaN=0  index=Date
targets_ml2    shape=(5433, 8)  NaN=0  index=Date

Files:


In [22]:
pd.read_parquet("../src/data/financial_dataset/processed/features_ml1.parquet")

,SPY_ret_1d,SPY_ret_5d,SPY_ret_21d,SPY_ret_63d,SPY_vol_5d,SPY_vol_21d,SPY_vol_63d,VIX,VIX_chg_5d,SPY_drawdown_252d,SPY_vs_ma200,T10Y2Y,SPY_IEF_corr_63d,BAA10Y
Date,,,,,,,,,,,,,,
2005-09-29,0.008137,0.010878,0.017614,0.033580,0.053652,0.095823,0.090121,12.24,-0.081770,-0.012330,0.029437,0.16,-0.020388,1.83
2005-09-30,0.003098,0.013176,0.008025,0.033747,0.052470,0.086253,0.090139,11.92,-0.080247,-0.009271,0.032445,0.16,-0.008970,1.81
2005-10-03,-0.003576,0.008389,0.005159,0.021843,0.068977,0.087278,0.089204,12.46,-0.044479,-0.012814,0.028588,0.18,0.032801,1.82
2005-10-04,-0.011256,-0.002715,-0.004367,0.018882,0.116356,0.095819,0.090524,13.20,0.034483,-0.023925,0.016873,0.16,0.043928,1.81
2005-10-05,-0.013117,-0.016767,-0.028786,0.001577,0.144508,0.095586,0.094104,14.55,0.152019,-0.036728,0.003467,0.15,0.002821,1.81
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-09-29,-0.001842,-0.011870,-0.004227,0.025882,0.084253,0.108357,0.110562,16.04,0.128783,-0.015147,0.067756,0.37,0.551095,1.46
2026-09-30,-0.002054,-0.006746,-0.003293,0.025161,0.072818,0.108100,0.110617,16.34,0.076416,-0.017170,0.064961,0.41,0.551096,1.47
2026-10-01,0.001783,-0.004158,0.005391,0.028340,0.076264,0.105430,0.110593,16.39,0.045948,-0.015417,0.066194,0.46,0.554906,1.49


In [23]:
pd.read_parquet("../src/data/financial_dataset/processed/features_ml2.parquet")

,SPY_vol_1d,SPY_vol_5d,SPY_vol_21d,SPY_vol_63d,IEF_vol_1d,IEF_vol_5d,IEF_vol_21d,IEF_vol_63d,LQD_vol_1d,LQD_vol_5d,LQD_vol_21d,LQD_vol_63d,VNQ_vol_1d,VNQ_vol_5d,VNQ_vol_21d,VNQ_vol_63d,SPY_ret_1d,SPY_ret_5d,SPY_ret_21d,SPY_ret_63d,IEF_ret_1d,IEF_ret_5d,IEF_ret_21d,IEF_ret_63d,LQD_ret_1d,LQD_ret_5d,LQD_ret_21d,LQD_ret_63d,VNQ_ret_1d,VNQ_ret_5d,VNQ_ret_21d,VNQ_ret_63d,VIX,VIX_chg_1d,VIX_chg_5d,SPY_IEF_corr_63d,SPY_LQD_corr_63d,SPY_VNQ_corr_63d,IEF_LQD_corr_63d,vol_ratio_SPY_IEF_21d,vol_ratio_VNQ_SPY_21d,vol_ratio_LQD_IEF_21d,mean_vol_21d
Date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2004-12-31,0.034066,0.052969,0.067641,0.103607,0.018894,0.048345,0.059928,0.054743,0.001707,0.052688,0.052682,0.055275,0.042002,0.090051,0.120032,0.153515,-0.002146,0.000829,0.018544,0.089064,0.001190,-0.001390,1.201034e-02,0.000426,0.000108,-0.002121,0.014517,0.010406,-0.002646,0.004083,0.027634,0.159979,13.29,0.058121,0.183437,-0.130388,-0.064665,0.482088,0.859846,1.128712,1.774535,0.879086,0.075071
2005-01-03,0.074866,0.062494,0.070370,0.104242,0.031767,0.034589,0.059204,0.054843,0.102478,0.049687,0.054316,0.056310,0.185271,0.115770,0.123718,0.155624,-0.004716,-0.001825,0.012891,0.084704,0.002001,0.004847,1.632802e-02,0.003477,0.006455,0.009721,0.024315,0.019931,-0.011671,-0.000179,0.024379,0.137315,14.08,0.059443,0.159802,-0.139407,-0.093701,0.496773,0.854028,1.188607,1.758098,0.917441,0.076902
2005-01-04,0.193980,0.081710,0.083234,0.102796,0.098855,0.062566,0.053799,0.055686,0.089095,0.073839,0.056542,0.056764,0.238594,0.149212,0.127409,0.155700,-0.012220,-0.019393,0.001185,0.053631,-0.006227,-0.001294,8.091688e-07,0.001240,-0.005612,0.002976,0.012857,0.018718,-0.015030,-0.018892,-0.004402,0.099865,13.98,-0.007102,0.165000,-0.034485,0.012860,0.493390,0.857456,1.547130,1.530728,1.050977,0.080246
2005-01-05,0.109538,0.067323,0.086672,0.103941,0.026275,0.061502,0.053704,0.055778,0.012795,0.071430,0.055989,0.056740,0.527701,0.228502,0.169795,0.170422,-0.006900,-0.027604,-0.005390,0.044614,0.001655,0.002366,-2.277565e-04,0.002544,0.000806,0.005314,0.010842,0.020183,-0.033242,-0.056885,-0.042072,0.058752,14.09,0.007868,0.212565,-0.042966,0.010686,0.506004,0.857801,1.613885,1.959037,1.042546,0.091540
2005-01-06,0.080706,0.101112,0.081910,0.104306,0.014991,0.054872,0.053802,0.055807,0.011374,0.067874,0.055604,0.056212,0.122293,0.242186,0.168305,0.170935,0.005084,-0.020804,0.009062,0.049372,0.000944,-0.000460,8.341553e-04,0.003491,0.000716,0.002439,0.009200,0.016692,0.007704,-0.054145,-0.022301,0.067116,13.58,-0.036196,0.081210,-0.040108,0.012609,0.509251,0.866178,1.522426,2.054763,1.033481,0.089905
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-09-29,0.029235,0.084253,0.108357,0.110562,0.014185,0.084357,0.060860,0.051571,0.009295,0.079751,0.069041,0.059522,0.010514,0.089203,0.100515,0.125335,-0.001842,-0.011870,-0.004227,0.025882,-0.000894,-0.018758,-3.315698e-02,-0.044340,-0.000586,-0.025502,-0.033005,-0.049710,-0.000662,-0.024619,-0.060930,-0.053042,16.04,-0.001867,0.128783,0.551095,0.578713,0.163669,0.956788,1.780438,0.927620,1.134422,0.084693
2026-09-30,0.032613,0.072818,0.108100,0.110617,0.024845,0.058389,0.060842,0.051483,0.035653,0.057858,0.069070,0.059535,0.157816,0.057006,0.101811,0.126240,-0.002054,-0.006746,-0.003293,0.025161,-0.001565,-0.009757,-3.352523e-02,-0.043512,-0.002246,-0.016460,-0.033905,-0.049851,-0.009941,-0.019902,-0.062553,-0.066232,16.34,0.018703,0.076416,0.551096,0.579079,0.171484,0.956884,1.776741,0.941826,1.135239,0.084956
2026-10-01,0.028309,0.076264,0.105430,0.110593,0.001776,0.049388,0.060719,0.051387,0.023304,0.047344,0.067866,0.059338,0.081471,0.056737,0.101881,0.123487,0.001783,-0.004158,0.005391,0.028340,-0.000112,-0.004348,-3.040169e-02,-0.044534,-0.001468,-0.010858,-0.030317,-0.052818,-0.005132,-0.021937,-0.066009,-0.082398,16.39,0.003060,0.045948,0.554906,0.583313,0.180881,0.95582